# **BankShield: Detección de Phishing Bancario con Deep Learning**

**Miembros del equipo**
 * Florencia Barrios
 * María Inés Martinote
 * Florencia Lucas

## **1. Introducción**

**Problema y motivación**

El fraude a través de mensajes de texto (Smishing) es una de las mayores amenazas para los usuarios de banca móvil. Los atacantes envían mensajes urgentes para robar credenciales. La motivación de este proyecto es crear un filtro automático que identifique estos mensajes maliciosos. Esto beneficia a los usuarios bancarios al prevenir estafas y a las instituciones financieras al reducir reportes de seguridad y mejorar la confianza del cliente.

**Dataset**

Utilizaremos el dataset de Kaggle: “SMS Spam Collection Dataset”. El cual contiene 5574 mensajes.
https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

Este conjunto de datos contiene miles de mensajes reales etiquetados como "spam" (fraude/no deseado) y "ham" (legítimo). El input será el texto del mensaje y el output será la probabilidad de que sea un fraude.

**Propuesta de Deep Learning**

Implementaremos un modelo de LLM (Large Language Model), por ejemplo utilizando DistilBERT. Utilizaremos la librería transformers de Hugging Face para realizar el entrenamiento.


**Importante**: Cambiar el entorno de ejecución a GPU!

**LIBRERÍAS**

In [ ]:
# 'transformers' es la librería de Hugging Face que contiene el modelo DistilBERT.
!pip install -q transformers[torch] datasets evaluate

In [ ]:
import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

## **2. Descripción de los datos y preparación**

### 2.1. Fuente y Etiquetado
El dataset contiene mensajes reales etiquetados como:
- **Ham (0):** Mensajes legítimos (ej. "Nos vemos en 5 min").
- **Spam (1):** Mensajes no deseados o fraudulentos (ej. "Gana 1000$ ahora").

### 2.2. Preparación y Limpieza
En ML tradicional, tendríamos que limpiar stop-words o hacer stemming. En Deep Learning con Transformers, **no es recomendable** hacer eso, ya que el modelo necesita la estructura completa de la frase para entender el contexto.

Lo que sí hacemos es:
1. Eliminar duplicados.
2. Renombrar columnas para claridad.
3. Dividir los datos en Entrenamiento (80%) y Prueba (20%).

**CARGA DEL DATASET**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Deep Learning/Proyecto final/spam.csv', encoding='latin1')
#df = pd.read_csv('/content/drive/MyDrive/DEEP LEARNING/spam.csv', encoding='latin1') #ine

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

**LIMPIEZA**

In [ ]:
# --- Exploración Inicial ---
print("Columnas originales:", df.columns.tolist())

In [ ]:
# --- Limpieza ---
# 1. Eliminamos columnas innecesarias (Unnamed 2, 3 y 4)
df = df.drop(columns=["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], errors='ignore')

In [ ]:
# 2. Renombramos v1 y v2 a nombres descriptivos
df = df.rename(columns={"v1": "label", "v2": "sms"})

In [ ]:
# 3. Convertimos etiquetas a números: ham -> 0, spam -> 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [ ]:
# 4. Eliminamos duplicados
print(f"Duplicados encontrados: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
# --- Visualización ---
plt.figure(figsize=(7, 5))
# Definimos los colores: 0 (Ham) -> Gris, 1 (Spam) -> Rojo
colores = {0: "silver", 1: "#d62728"}
sns.countplot(x='label', data=df, palette=colores, hue='label', legend=False)

plt.title('Distribución de Mensajes: Legítimos vs. Spam', fontsize=14)
plt.xlabel('Categoría', fontsize=12)
plt.ylabel('Cantidad de Mensajes', fontsize=12)
plt.xticks([0, 1], ['Legítimo', 'Spam'])
sns.despine() # Limpia los bordes del gráfico para que se vea más profesional
plt.show()

In [ ]:
print(f"Total de mensajes después de limpiar: {len(df)}")
print(df.head())

**PREPARACIÓN PARA DEEP LEARNING**

In [ ]:
# Dividimos en entrenamiento (80%) y prueba (20%)
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Convertimos de Pandas DataFrame a formato 'datasets' de Hugging Face
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)
dataset = DatasetDict({'train': train_dataset, 'test': test_dataset})

In [ ]:
len(df), len(df_train), len(df_test)

## **3. Solución Propuesta: DistilBERT**

Nuestra solución utiliza **DistilBERT**, un modelo de Deep Learning basado en la arquitectura **Transformer**. A diferencia de los modelos estadísticos que cuentan palabras, este modelo "entiende" la semántica.

### ¿Cómo funciona?
1. **Transfer Learning y Fine-tuning:**
    - **¿Qué es?** Es como si tomáramos a un estudiante que ya sabe leer perfectamente (modelo pre-entrenado en Wikipedia) y le diéramos un curso de especialización en seguridad bancaria (**Fine-tuning**).
   - **¿Cómo se hace?** Cargamos los pesos de una red neuronal gigante y la entrenamos un poco más con nuestros mensajes de SMS. En lugar de aprender a leer desde cero, el modelo solo ajusta sus conexiones para reconocer patrones de fraude.
   
2. **Mecanismo de Atención (Self-Attention):**
    - Es la capacidad del modelo para ponderar la importancia de diferentes palabras en una frase simultáneamente.
   - **Ejemplo Práctico:**  *"Su **tarjeta** ha sido **bloqueada** por seguridad, verifique **aquí**."*
   - Un modelo viejo vería palabras sueltas. DistilBERT crea un "mapa de relaciones" donde la palabra **tarjeta** está fuertemente conectada con **bloqueada**, y ambas se conectan con el enlace **aquí**. El modelo "pone su atención" en esta combinación que transmite urgencia, reconociendo el patrón de estafa aunque cambien las palabras.

3. **Tokenización:**
    - El modelo descompone el texto en unidades mínimas llamadas tokens (pedazos de palabras). Esto permite que, si el atacante escribe mal una palabra a propósito (ej. "cu€nta"), el modelo aún pueda reconocer la raíz y no romperse.



**TOKENIZACIÓN**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["sms"], truncation=True, padding=True)

tokenized_data = dataset.map(preprocess_function, batched=True)

**VISUALIZACIÓN DE LA TOKENIZACIÓN (Ejemplos)**

In [ ]:
print("\n--- EJEMPLO DE TOKENIZACIÓN ---")
mensaje_ejemplo = "Your bank acc##ount is locked! Click: http://bit.ly/fake"

# 1. Ver cómo se rompe en tokens
tokens = tokenizer.tokenize(mensaje_ejemplo)
print(f"1. Tokens individuales:\n{tokens}")

# 2. Ver cómo se convierte en IDs numéricos
ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"\n2. IDs numéricos (Tokens -> Números):\n{ids}")

# 3. Ver el resultado final (incluyendo tokens especiales [CLS] y [SEP])
input_completo = tokenizer(mensaje_ejemplo)
print(f"\n3. Input final para el modelo (con caracteres especiales):\n{input_completo['input_ids']}")
print(f"\n4. Decodificación de vuelta a texto:\n{tokenizer.decode(input_completo['input_ids'])}")


**CONFIGURACIÓN DEL MODELO**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

**Interpretación del código:**

**1. AutoModelForSequenceClassification**

Esta es una clase de la librería **Hugging Face** diseñada específicamente para tareas de clasificación de texto.

Esta clase le dice al sistema: "Toma un modelo de lenguaje y agrégale una capa final de clasificación al final de su estructura".

**2. .from_pretrained("distilbert-base-uncased")**

Aquí es donde se aplica el **Transfer Learning**:

No empezamos de cero: Estamos descargando los "**pesos**" (el conocimiento) de **DistilBERT**, un modelo que ya fue entrenado por Google en millones de frases de Wikipedia y libros.

"**distilbert-base-uncased**": Significa que es la versión ligera (Distil) y que no le importa si las letras son mayúsculas o minúsculas (Uncased).

**3. num_labels=2**

Esto define la estructura de la "salida" (output) del modelo:

Como nuestro problema es binario (0 para Legítimo, 1 para Fraude), le indicamos que la última capa debe tener exactamente 2 neuronas.

Cada neurona dará una puntuación (logit). La que tenga la puntuación más alta será la predicción final.

**Interpretación del mensaje de carga:**

- UNEXPECTED: Son capas del modelo original (Masked LM) que no sirven para clasificar y se descartan.
- MISSING: Son las nuevas capas de clasificación que hemos creado (para 2 etiquetas).
Están vacías y 'aprenderán' durante el entrenamiento en la Sección 6.

## **4. Métricas de Evaluación**

Para este problema, el **Accuracy** no es suficiente porque los datos están desbalanceados (hay mucho más "Ham" que "Spam"). Si el modelo dijera siempre que todo es "Ham", tendría un 86% de accuracy pero sería inútil.

Por eso usamos:
- **Accuracy:** Porcentaje total de aciertos.
- **F1-Score:** El balance entre Precisión (¿cuántos de los que dije que eran fraude realmente lo eran?) y Recall (¿cuántos de todos los fraudes reales logré atrapar?). **Esta es nuestra métrica principal.**


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

**ENTRENAMIENTO**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir='./logs',
    report_to="none"
)

**Interpretación:**

**learning_rate=0.00002 (Tasa de aprendizaje):**

Es el parámetro más importante. Define qué tan grandes son los "pasos" que da el modelo para ajustar sus pesos. Usamos un valor muy pequeño (0.00002) porque estamos haciendo *Fine-tuning*: no queremos que el modelo olvide lo que ya sabe de lenguaje, solo que se ajuste suavemente a los SMS bancarios.

**per_device_train_batch_size=16:**

El modelo no mira todos los mensajes a la vez, sino que los procesa en grupos (lotes) de 16. Esto ayuda a que la memoria de la GPU no se sature y que el aprendizaje sea más estable.

**num_train_epochs=3:**

Es la cantidad de veces que el modelo revisará el dataset completo. Como **DistilBERT** es muy potente, con 3 o 4 pasadas ya alcanza su máximo rendimiento.

**weight_decay=0.01:**

Es una técnica de **regularización**. Básicamente, penaliza los pesos demasiado grandes para evitar que el modelo se vuelva "rígido" o aprenda los datos de memoria (ayuda directamente a prevenir el **Overfitting**).

**eval_strategy="epoch" y load_best_model_at_end=True:**

Esto es como un "seguro de calidad". Le decimos al modelo que se examine al final de cada época y que, cuando termine todo el proceso, descarte las versiones mediocres y se quede automáticamente con la versión que obtuvo mejores resultados en el set de prueba.

*El próximo paso demora aprox. 5 min.*

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("\nIniciando entrenamiento...")
trainer.train()

**VISUALIZACIÓN DE MÉTRICAS**

In [ ]:
# Extraemos los logs del historial del trainer
history = trainer.state.log_history

results = trainer.evaluate()
print(f"\nResultados finales:")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Score: {results['eval_f1']:.4f}")
# Filtramos los logs para obtener pérdida de evaluación
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
epochs = range(1, len(eval_loss) + 1)

# Graficamos la Matriz de Confusión y la Curva de Pérdida
predictions = trainer.predict(tokenized_data["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Matriz de Confusión
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legítimo", "Fraude"])
disp.plot(cmap=plt.cm.Reds, ax=ax[0])
ax[0].set_title("Matriz de Confusión Final")

# Subplot 2: Curva de Pérdida (Loss Curve) con Eje Normalizado
ax[1].plot(epochs, eval_loss, 'o-', color='blue', linewidth=2, label='Pérdida de Evaluación')
ax[1].set_title("Curva de Pérdida por Época (Escala 0.0 a 0.1)")
ax[1].set_xlabel("Época")
ax[1].set_ylabel("Loss")

# AJUSTE DEL EJE Y: Para que se vea que la pérdida es muy baja en general
# Ponemos el límite de 0 a 0.1 para dar perspectiva de "Loss baja"
ax[1].set_ylim(0, 0.1)

ax[1].legend()
ax[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## **5. Resultados y Discusión**

### 5.1. Análisis de Métricas
El modelo alcanza resultados excepcionales con un Accuracy de **99%** y un F1-Score cercano a **0.96**. La Matriz de Confusión muestra una capacidad de detección casi perfecta, con solo 4 falsos positivos y 6 falsos negativos sobre una muestra de más de 1000 mensajes.

### 5.2. Curvas de Aprendizaje e Interpretación
Al observar la **Curva de Pérdida (Loss)**, notamos un comportamiento interesante:
1. **Convergencia Rápida:** La pérdida cae drásticamente de la época 1 a la 2, indicando que DistilBERT adapta su conocimiento previo al dominio bancario de forma eficiente.
2. **Optimización de Épocas:** Mantener el entrenamiento en 3 épocas evita el **overfitting**, permitiendo que el modelo generalice ante nuevos tipos de fraude sin memorizar el dataset de entrenamiento.


### 5.3. Limitaciones y Overfitting
- **Overfitting:** Para evitar que el modelo solo memorice los datos, usamos *Weight Decay* y pocas épocas de entrenamiento. La curva de pérdida de validación se mantuvo cercana a la de entrenamiento.
- **Dificultades:** El dataset original está en inglés. Para un banco local, el modelo debería entrenarse con datos en español, aunque DistilBERT tiene versiones multilingües.


**PRUEBA EN TIEMPO REAL**

In [ ]:
def predict_message(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    prediction = torch.argmax(logits, dim=-1).item()
    return "  SPAM ⚠️" if prediction == 1 else "LEGÍTIMO ✅"

print("\n--- DEMO DE PREDICCIÓN ---")
test_msgs = [
    "Your bank acc##ount has been locked. Click here to verify: http://bit.ly/fake-bank",
    "Hey! Are we still meeting for lunch today?",
    "URGENT: Your credit card showed a $500 transaction. Call us immediately."
]

for msg in test_msgs:
    print(f"Msg: {msg} -> Resultado: {predict_message(msg)}")

## **6. CONCLUSIONES**

La implementación de BankShield demuestra que el uso de Deep Learning, y específicamente de arquitecturas basadas en Transformers como **DistilBERT**, supera significativamente las limitaciones de los enfoques tradicionales de seguridad financiera. A lo largo del proyecto, hemos llegado a las siguientes conclusiones clave:

1. **Eficacia del Transfer Learning:** El proceso de *Fine-tuning* permitió que un modelo pre-entrenado en lenguaje general se especializara en la detección de estafas bancarias en cuestión de minutos. El hecho de alcanzar un **Accuracy superior al 99%** con solo 4 épocas de entrenamiento confirma que no es necesario entrenar modelos desde cero para obtener resultados de nivel industrial.

2. **Superioridad de la Atención sobre la Frecuencia:** Al analizar los resultados y la inspección detallada de mensajes, comprobamos que el **Mecanismo de Atención** es vital. El modelo es capaz de identificar la relación semántica entre términos de urgencia ("bloqueada", "urgente") y elementos de acción ("click aquí", "enlace"), ignorando variaciones ortográficas que suelen confundir a los filtros basados en reglas.

3. **Robustez y Estabilidad:** La visualización de la **Curva de Pérdida** revela un aprendizaje estable. La mínima fluctuación observada en las etapas finales no representa inestabilidad, sino la madurez del modelo al operar con márgenes de error extremadamente bajos. Esto asegura que BankShield es una herramienta fiable que no sufre de *overfitting* severo.

4. **Impacto en el Negocio Bancario:** Desde una perspectiva operativa, el uso del **F1-Score** como métrica principal garantiza un equilibrio entre seguridad y experiencia de usuario. Minimizar los Falsos Positivos evita bloqueos innecesarios a clientes legítimos, mientras que la reducción drástica de Falsos Negativos protege directamente el capital de la institución.

En conclusión, BankShield representa una solución moderna, escalable y de alta precisión que puede ser integrada fácilmente en infraestructuras bancarias reales para combatir el fraude digital de manera proactiva.


## **Probando en otra base: (The Phishing Email Dataset.)**

In [ ]:
import pandas as pd
dataset = pd.read_csv('/content/drive/MyDrive/DEEP LEARNING/phishing_email.csv', encoding='latin1')

Importamos librerias

In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Define el nombre del archivo
MODEL_PATH = 'spam_bert_model.pth'

# Guardar solo los pesos (state_dict) es lo más eficiente
torch.save(model.state_dict(), MODEL_PATH)

print(f"✅ Modelo guardado exitosamente como {MODEL_PATH}")
# Opcional: Descargarlo automáticamente a tu PC
from google.colab import files
files.download(MODEL_PATH)

Analisis y limpieza de datos

In [ ]:
#mostrame las primeras lineas de estas dataset
dataset.head ()

In [ ]:
#cuantos valores tiene el dataset
dataset.shape

In [ ]:
#nombres de las columnas
dataset.columns

In [ ]:
# valores nulos
print(dataset.isnull().sum())

In [ ]:
# valores duplicados
print(dataset.duplicated().sum())

In [ ]:
# eliminamos valores duplicados
dataset.drop_duplicates(inplace=True)

In [ ]:
#chequeamos con que dataset nos quedamos
dataset.shape

In [ ]:
#cuales son los nombres de las columnas
dataset.columns

In [ ]:
# definimos df_phsinig como dataset
df_phishing = dataset

In [ ]:
# =====================================================================
# 1. FUNCIÓN PARA PREPROCESAR Y CARGAR DATOS
# =====================================================================
def preprocess_email(text):
    text_combined = str(text).lower()
    text_combined = re.sub(r'<[^>]+>', '', text_combined) # Eliminar etiquetas HTML
    text_combined = re.sub(r'http\S+', 'http_link', text_combined) # Reemplazar URLs
    text_combined = re.sub(r'[^a-z\s]', '', text_combined) # Eliminar caracteres especiales y números
    text_combined = re.sub(r'\s+', ' ', text_combined).strip() # Eliminar espacios múltiples
    return text_combined

def load_new_dataset(file_path):
    if not os.path.exists(file_path):
        print(f"⚠️ Archivo {file_path} no encontrado. Creando datos de prueba para demostración.")
        data = {
            'text': [
                "Your account has been compromised, click here to reset",
                "Meeting at 5pm in the conference room",
                "Urgent: verify your bank details now",
                "The report you requested is attached",
                "Win a free iPhone by clicking this link!"
            ],
            'label': [1, 0, 1, 0, 1] # 1: Phishing/Spam, 0: Ham/Safe
        }
        df = pd.DataFrame(data)
    else:
        df = pd.read_csv(file_path, encoding='latin1')

    # Standardize column names and label format
    if 'Email Type' in df.columns and 'Email Text' in df.columns:
        df['label'] = df['Email Type'].map({'Phishing Email': 1, 'Safe Email': 0})
        df = df.rename(columns={'Email Text': 'text'})
    elif 'text_combined' in df.columns and 'label' in df.columns:
        df = df.rename(columns={'text_combined': 'text'})
    else:
        print("❓ Columnas no reconocidas. Intentando detección automática...")
        df.columns = [col.strip() for col in df.columns]

        text_col_candidate = None
        label_col_candidate = None

        # Try to find a text column
        text_cols = [c for c in df.columns if 'text' in c.lower() or 'message' in c.lower() or 'email' in c.lower()]
        if text_cols:
            text_col_candidate = text_cols[0]
        else:
            object_cols = df.select_dtypes(include=['object']).columns.tolist()
            if object_cols:
                text_col_candidate = object_cols[0]

        # Try to find a label column
        label_cols = [c for c in df.columns if 'label' in c.lower() or 'type' in c.lower() or 'v1' in c.lower()]
        if label_cols:
            label_col_candidate = label_cols[0]

        if text_col_candidate and label_col_candidate:
            df = df.rename(columns={text_col_candidate: 'text', label_col_candidate: 'label'})
            if df['label'].dtype == 'object':
                 df['label'] = pd.factorize(df['label'])[0]
        else:
            raise ValueError("No se pudieron identificar las columnas de texto y etiqueta automáticamente.")

    # Apply email preprocessing here
    if 'text' in df.columns:
        df['text'] = df['text'].apply(preprocess_email)

    # Clean: remove nulls and reset index
    df = df[['text', 'label']].dropna().reset_index(drop=True)
    print(f"✅ Dataset cargado y preprocesado: {len(df)} correos encontrados.")
    return df


In [ ]:
# =====================================================================
# 2. FUNCIÓN PARA CARGAR EL MODELO ENTRENADO
# =====================================================================
def load_trained_model(model_path):
    # Re-instanciar el tokenizer y el modelo con la configuración original
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

    # Mover el modelo a la CPU si se guardó desde la GPU o para un entorno sin GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(model_path, map_location=device))

    # Poner el modelo en modo de evaluación
    model.eval()

    # Mover el modelo al dispositivo correcto (GPU si está disponible)
    model.to(device)

    print(f"✅ Modelo cargado exitosamente desde {model_path} en {device}")
    return model, tokenizer


In [ ]:
# =====================================================================
# 3. FUNCIÓN PARA EVALUAR EN NUEVOS DATOS
# =====================================================================
def evaluate_on_new_data(model, tokenizer, df_new_data):
    # Aplicar preprocesamiento a la columna de texto del nuevo dataset
    df_new_data_processed = df_new_data.copy()

    # Asegurarse de que la columna de texto se llama 'text' para el tokenizador
    if 'text_combined' in df_new_data_processed.columns:
        df_new_data_processed['text'] = df_new_data_processed['text_combined'].apply(preprocess_email)
    elif 'text' in df_new_data_processed.columns:
        df_new_data_processed['text'] = df_new_data_processed['text'].apply(preprocess_email)
    else:
        raise ValueError("No se encontró la columna 'text_combined' ni 'text' en el DataFrame.")

    # Crear un dataset de Hugging Face
    new_dataset = Dataset.from_pandas(df_new_data_processed[['text', 'label']])

    # Tokenizar los nuevos datos
    def tokenize_function(examples):
        # Explicitly pad to max_length of the model (512 for DistilBERT)
        return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=tokenizer.model_max_length)

    tokenized_new_data = new_dataset.map(tokenize_function, batched=True)

    # Mover el modelo a la GPU si está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Preparar para la predicción
    input_ids = torch.tensor(tokenized_new_data["input_ids"])
    attention_mask = torch.tensor(tokenized_new_data["attention_mask"])
    labels = torch.tensor(new_dataset["label"])

    # Crear un DataLoader para la predicción
    prediction_dataset = TensorDataset(input_ids, attention_mask, labels)
    prediction_dataloader = DataLoader(prediction_dataset, batch_size=16)

    true_labels = []
    predicted_labels = []

    model.eval()
    with torch.no_grad():
        for batch in prediction_dataloader:
            batch = tuple(t.to(device) for t in batch)
            b_input_ids, b_input_mask, b_labels = batch

            outputs = model(b_input_ids, attention_mask=b_input_mask)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1).flatten()

            true_labels.extend(b_labels.cpu().numpy())
            predicted_labels.extend(preds.cpu().numpy())

    print("✅ Evaluación en el nuevo dataset completada.")
    return true_labels, predicted_labels

In [ ]:
# =====================================================================
# 4. FUNCIÓN PARA GRAFICAR RESULTADOS (Añadida)
# =====================================================================
def plot_results(y_true, y_pred):
    print(f"\n--- MÉTRICAS FINALES ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print("\nReporte de Clasificación:")
    print(classification_report(y_true, y_pred, target_names=["Legítimo (0)", "Phishing (1)"]))

    # Matriz de confusión
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legítimo", "Phishing"])

    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')
    ax.set_title("Matriz de Confusión - Evaluación Externa")
    plt.show()


Demora aprox. 30' minutos

In [ ]:
# =====================================================================
# 5. BLOQUE DE EJECUCIÓN PRINCIPAL
# =====================================================================
if __name__ == "__main__":
    # Como el dataframe "dataset" ya existe en la memoria de tu Notebook, lo usamos directamente.
    # Asegúrate de haber ejecutado la celda donde importas phishing_email.csv antes de esto.
    df_test = dataset

    print("⏳ Cargando modelo guardado...")
    model, tokenizer = load_trained_model('spam_bert_model.pth')

    print("⏳ Iniciando la evaluación. Esto puede demorar entre 20 y 30 minutos...")
    true_labels, predicted_labels = evaluate_on_new_data(model, tokenizer, df_test)

    print("📊 Generando reporte y gráficas...")
    plot_results(true_labels, predicted_labels)

**Análisis de Generalización:**

**Dificultades:** Los correos de phishing son más largos que los SMS del dataset original.

**Limitaciones:** El modelo BERT tiene un límite de 512 tokens; correos muy largos son truncados.

**Overfitting:** Si la precisión bajó más de un 15% respecto al set de validación original, indica que el modelo aprendió patrones muy específicos de la primera base que no existen en la segunda.

1. Análisis de la Matriz de Confusión

El gran problema (Falsos Positivos - Arriba a la derecha): De los 39,233 correos que eran legítimos (seguros), el modelo se equivocó y marcó 37,522 como Phishing. Esto significa que casi bloquea todos los correos normales de los usuarios.

Aciertos en Phishing (Verdaderos Positivos - Abajo a la derecha): Logró identificar correctamente 36,611 correos que sí eran Phishing.

Aciertos en Legítimos (Verdaderos Negativos - Arriba a la izquierda): Solo logró identificar correctamente 1,711 correos legítimos.

Peligros que dejó pasar (Falsos Negativos - Abajo a la izquierda): Dejó pasar 6,234 correos de Phishing etiquetándolos como seguros.




2. Análisis de las Métricas
Accuracy (Exactitud - 46.69%): El modelo acertó menos de la mitad de las veces. En problemas binarios (2 opciones), un 46% es prácticamente peor que lanzar una moneda al azar. En tu entrenamiento original con SMS tenías un 99.13%, por lo que la caída es drástica.

Recall (Sensibilidad):

Para Phishing es del 0.85 (85%): Encontró la mayoría de los correos maliciosos.

Para Legítimos es del 0.04 (4%): Solo reconoció el 4% de los correos buenos.

Precision (Precisión):

Para Phishing es del 0.49 (49%): Como marcó casi todo el dataset como "Phishing", cuando el modelo dice que un correo es malo, solo tiene razón la mitad de las veces.


3. Conclusión:

Overfitting al formato SMS: El modelo aprendió a ser un experto leyendo mensajes de texto cortos, con abreviaturas y enlaces directos (Kaggle SMS Spam). Al ver correos electrónicos (que tienen encabezados, firmas, texto largo, html residual y vocabulario formal), el modelo no supo qué hacer y, ante la duda, clasificó casi todo como amenaza.

Límite de Tokens: Como mencionamos, DistilBERT corta el texto a los 512 tokens. Muchos correos electrónicos superan este límite, por lo que el modelo probablemente tomó decisiones "leyendo" solo el principio del correo.

**Podriamos llegar a mejroar este modelo haciendole un segundo proceso de Fine-Tuning (entrenamiento) utilizando este nuevo dataset de correos.**